<a href="https://colab.research.google.com/github/lerkalarionova2018-rgb/nlp-homeworks/blob/main/%D0%9B%D0%B0%D1%80%D0%B8%D0%BE%D0%BD%D0%BE%D0%B2%D0%B0_%22w2v_hw_ipynb%22.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

В этом практикуме мы рассмотрим работу с библиотекой **Gensim** для работы с векторными представлениями текста

Мы рассмотрим
- **Word2Vec** - векторные представления слов
- **FastText** - улучшенные представления с учетом морфологии  
- **Doc2Vec** - векторные представления документов


In [1]:
!pip install gensim

import gensim
import gensim.downloader as api
from gensim.models import Word2Vec, FastText, Doc2Vec
from gensim.models.doc2vec import TaggedDocument
import numpy as np

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 35.8 MB/s eta 0:00:00


## Часть 1: Word2Vec

### Что такое Word2Vec?

Word2Vec преобразует слова в векторы чисел так, что семантически похожие слова оказываются близко в векторном пространстве.

**Два основных алгоритма:**
- **CBOW** - предсказывает слово по контексту
- **Skip-gram** - предсказывает контекст по слову

**Загрузка предобученной модели**

In [2]:
w2v_model = api.load('glove-wiki-gigaword-100')

print(f"Размер словаря: {len(w2v_model.key_to_index)}")
print(f"Размерность векторов: {w2v_model.vector_size}")

[==================================================] 100.0% 128.1/128.1MB downloaded
Размер словаря: 400000
Размерность векторов: 100


Найдите документацию `gensim`: какие датасеты кроме `glove-wiki-gigaword-100` доступны в библиотеке?

Выберите 3 датасета и кратко опишите их (источник данных, примерный объем, зачем такой датасет может использоваться)

**Базовые операции с векторами**

In [3]:
# Получаем вектор слова
vector = w2v_model['computer']
print(f"Вектор слова 'computer': {vector[:5]}...")  # Показываем первые 5 чисел

# Вычисляем схожесть между словами
similarity = w2v_model.similarity('computer', 'laptop')
print(f"Схожесть 'computer' и 'laptop': {similarity:.4f}")

Вектор слова 'computer': [-0.16298   0.30141   0.57978   0.066548  0.45835 ]...
Схожесть 'computer' и 'laptop': 0.7024


**Поиск похожих слов**

In [4]:
# Находим похожие слова
similar_words = w2v_model.most_similar('python', topn=5)
print("Слова, похожие на 'python':")
for word, score in similar_words:
    print(f"  {word}: {score:.4f}")

Слова, похожие на 'python':
  monty: 0.6886
  php: 0.5865
  perl: 0.5784
  cleese: 0.5447
  flipper: 0.5113


*Ваш ответ здесь*

In [8]:
# Просмотр первых 3 датасетов
import pprint
api_info = api.info()
print("Первые 3 датасета:")
pprint.pprint(list(api_info['models'].keys())[:3])

Первые 3 датасета:
['fasttext-wiki-news-subwords-300',
 'conceptnet-numberbatch-17-06-300',
 'word2vec-ruscorpora-300']


1. 'fasttext-wiki-news-subwords-300'
  - Источник: Wikipedia 2017 и новостные статьи
  - Объем: около 1 млн слов
  - Применение: работа с редкими словами и опечатками благодаря использованию n-grams

2. 'conceptnet-numberbatch-17-06-300'
  - Источник: ConceptNet (объединение Wiktionary, WordNet и др.)
  - Объем: около 500 тыс. терминов
  - Применение: поиск семантических аналогий

3. 'word2vec-ruscorpora-300'
  - Источник: Национальный корпус русского языка и русская Wikipedia
  - Объем: около 185 тыс. русских слов
  - Применение: обработка русскоязычных текстов, поиск семантически близких слов в русском языке

**Задание**

1. Загрузите любой датасет из gensim на ваш выбор

In [11]:
test_model = api.load('glove-twitter-25')

print(f"Размер словаря: {len(test_model.key_to_index)}")
print(f"Размерность векторов: {test_model.vector_size}")

[==================================================] 100.0% 104.8/104.8MB downloaded
Размер словаря: 1193514
Размерность векторов: 25


2. Напишите функцию, которая принимает на вход любое слово и вовращает 10 наиболее близких по вектору слов

In [12]:
def find_similar_words(word, model, topn=10):
    try:
        similar = model.most_similar(word, topn=topn)

        print(f"Слова, похожие на '{word}':")

        for w, score in similar:
            print(f"  {w}: {score:.4f}")

    except KeyError:
        print(f"Слово '{word}' отсутствует в словаре модели")

# Пример
find_similar_words('music', test_model)

Слова, похожие на 'music':
  dance: 0.9096
  song: 0.9092
  story: 0.9006
  mix: 0.8984
  songs: 0.8969
  part: 0.8955
  world: 0.8949
  dream: 0.8944
  movie: 0.8855
  album: 0.8850


3. Обучите модель Word2Vec на тестовом датасете из ячейки ниже

Примените следующие настройки:

- размер вектора: 50
- размер окна: 3
- минимальная частота слова: 1
- потоков: 2
- использовать skip-gram

In [13]:
cooking_sentences = [
    ['варить', 'суп', 'овощи', 'морковь', 'картофель'],
    ['жарить', 'курица', 'сковорода', 'масло', 'специи'],
    ['печь', 'хлеб', 'мука', 'дрожжи', 'духовка'],
    ['резать', 'овощи', 'салат', 'помидоры', 'огурцы'],
    ['смешивать', 'ингредиенты', 'тесто', 'яйца', 'молоко'],
    ['варить', 'паста', 'вода', 'соль', 'соус'],
    ['гриль', 'мясо', 'овощи', 'уголь', 'барбекю'],
    ['тушить', 'говядина', 'горшок', 'вино', 'травы'],
    ['запекать', 'рыба', 'лимон', 'духовка', 'фольга'],
    ['готовить', 'завтрак', 'яичница', 'бекон', 'тост'],
    ['месить', 'тесто', 'пирог', 'начинка', 'яблоки'],
    ['кипятить', 'вода', 'чай', 'кофе', 'чашка'],
    ['мариновать', 'мясо', 'соус', 'специи', 'холодильник'],
    ['взбивать', 'сливки', 'сахар', 'десерт', 'торт'],
    ['парить', 'овощи', 'здоровое', 'питание', 'брокколи']
]

In [14]:
model = Word2Vec(
    sentences=cooking_sentences,
    vector_size=50,
    window=3,
    min_count=1,
    workers=2,
    sg=1
)

print("Модель Word2Vec успешно обучена!")

Модель Word2Vec успешно обучена!


In [15]:
print(f"Слова в словаре: {list(model.wv.key_to_index.keys())[:10]}...")

Слова в словаре: ['овощи', 'мясо', 'соус', 'вода', 'тесто', 'духовка', 'специи', 'варить', 'брокколи', 'питание']...


4. Проверьте модель

In [16]:
# Проверяем похожие слова в кулинарной тематике
try:
    similar = model.wv.most_similar('варить', topn=5)
    print("Слова, похожие на 'варить':")
    for word, score in similar:
        print(f"  {word}: {score:.4f}")
except KeyError:
    print("Слово 'варить' не найдено в словаре")

Слова, похожие на 'варить':
  вино: 0.2398
  ингредиенты: 0.2172
  хлеб: 0.1938
  брокколи: 0.1846
  кипятить: 0.1711


In [17]:
print("\nСлова, похожие на 'духовка':")
try:
    similar = model.wv.most_similar('духовка', topn=5)
    for word, score in similar:
        print(f"  {word}: {score:.4f}")
except KeyError:
    print("Слово 'духовка' не найдено в словаре")

# Найдите слова, похожие на "овощи"
print("\nСлова, похожие на 'овощи':")
try:
    similar = model.wv.most_similar('овощи', topn=5)
    for word, score in similar:
        print(f"  {word}: {score:.4f}")
except KeyError:
    print("Слово 'овощи' не найдено в словаре")


Слова, похожие на 'духовка':
  ингредиенты: 0.3199
  десерт: 0.3064
  холодильник: 0.2705
  питание: 0.2243
  пирог: 0.2142

Слова, похожие на 'овощи':
  мариновать: 0.2716
  хлеб: 0.2691
  гриль: 0.2546
  фольга: 0.2409
  сахар: 0.2108


## Часть 2: FastText

FastText улучшает Word2Vec, рассматривая слова как наборы символов (n-грамм). Это позволяет работать с редкими словами и опечатками

5. Обучите FastText на корпусе текстов из пункта 3. Используйте код ниже

In [19]:
ft_model = FastText(
    sentences=cooking_sentences,
    vector_size=50,
    window=3,
    min_count=1,
    workers=2
)

print("FastText модель обучена!")
print(f"Слова в словаре: {list(ft_model.wv.key_to_index.keys())[:10]}...")
print(f"Всего слов в словаре: {len(ft_model.wv.key_to_index)}")

FastText модель обучена!
Слова в словаре: ['овощи', 'мясо', 'соус', 'вода', 'тесто', 'духовка', 'специи', 'варить', 'брокколи', 'питание']...
Всего слов в словаре: 65


6. Найдите слова, похожие на "варить", "духовка" и "овощи" с помощью обученной модели. Используйте код из пункта 4

In [20]:
print("Слова, похожие на 'варить' (FastText):")
try:
    similar = ft_model.wv.most_similar('варить', topn=5)
    for word, score in similar:
        print(f"  {word}: {score:.4f}")
except KeyError:
    print("Слово 'варить' не найдено в словаре")

print("\nСлова, похожие на 'духовка' (FastText):")
try:
    similar = ft_model.wv.most_similar('духовка', topn=5)
    for word, score in similar:
        print(f"  {word}: {score:.4f}")
except KeyError:
    print("Слово 'духовка' не найдено в словаре")

print("\nСлова, похожие на 'овощи' (FastText):")
try:
    similar = ft_model.wv.most_similar('овощи', topn=5)
    for word, score in similar:
        print(f"  {word}: {score:.4f}")
except KeyError:
    print("Слово 'овощи' не найдено в словаре")

Слова, похожие на 'варить' (FastText):
  жарить: 0.5353
  парить: 0.4805
  месить: 0.3541
  тушить: 0.3405
  специи: 0.2622

Слова, похожие на 'духовка' (FastText):
  взбивать: 0.4565
  лимон: 0.3561
  салат: 0.3050
  курица: 0.3041
  тост: 0.2944

Слова, похожие на 'овощи' (FastText):
  жарить: 0.2960
  фольга: 0.2574
  морковь: 0.2297
  соус: 0.2172
  торт: 0.2094


7. Сравните модели

Дана функция для сравнения Word2Vec и FastText

Придумайте 3 слова с опечатками и проверьте, найдет ли их FastText и Word2Vec

In [22]:
def compare_models(word):
    """Сравнивает представления слова в разных моделях"""
    print(f"\nСравнение для слова: '{word}'")

    # Word2Vec
    try:
        w2v_similar = model.wv.most_similar(word, topn=2)
        print(f"  Word2Vec: {[w for w, _ in w2v_similar]}")
    except KeyError:
        print(f"  Word2Vec: слово не найдено")

    # FastText
    try:
        ft_similar = ft_model.wv.most_similar(word, topn=2)
        print(f"  FastText: {[w for w, _ in ft_similar]}")
    except KeyError:
        print(f"  FastText: слово не найдено")

# Сравниваем для разных слов
compare_models('ворить')
compare_models('овоши')
compare_models('ингридиенты')


Сравнение для слова: 'ворить'
  Word2Vec: слово не найдено
  FastText: ['жарить', 'ингредиенты']

Сравнение для слова: 'овоши'
  Word2Vec: слово не найдено
  FastText: ['чай', 'яичница']

Сравнение для слова: 'ингридиенты'
  Word2Vec: слово не найдено
  FastText: ['ингредиенты', 'тушить']


## Часть 3: Doc2Vec

Doc2Vec расширяет Word2Vec для создания векторных представлений целых документов (предложений, абзацев, статей)

In [23]:
# Создаем размеченные документы
documents = [
    "machine learning is interesting",
    "deep learning uses neural networks",
    "python programming for data science",
    "artificial intelligence is amazing",
    "computer vision processes images"
]

# Преобразуем в формат TaggedDocument
tagged_docs = []
for i, doc in enumerate(documents):
    tokens = doc.split()
    tagged_doc = TaggedDocument(words=tokens, tags=[f"doc_{i}"])
    tagged_docs.append(tagged_doc)

print("Размеченные документы:")
for doc in tagged_docs[:3]:
    print(f"  Слова: {doc.words}")
    print(f"  Тег: {doc.tags}")

Размеченные документы:
  Слова: ['machine', 'learning', 'is', 'interesting']
  Тег: ['doc_0']
  Слова: ['deep', 'learning', 'uses', 'neural', 'networks']
  Тег: ['doc_1']
  Слова: ['python', 'programming', 'for', 'data', 'science']
  Тег: ['doc_2']


In [24]:
# Обучаем Doc2Vec
doc_model = Doc2Vec(
    documents=tagged_docs,
    vector_size=50,
    window=3,
    min_count=1,
    workers=2,
    epochs=20
)

print("Doc2Vec модель обучена!")
print(f"Количество документов: {len(doc_model.dv.key_to_index)}")

Doc2Vec модель обучена!
Количество документов: 5


In [25]:
# Получаем вектор документа
doc_vector = doc_model.dv["doc_0"]
print(f"Вектор документа doc_0: {doc_vector[:5]}...")

# Находим похожие документы
similar_docs = doc_model.dv.most_similar("doc_0", topn=2)
print("\nДокументы, похожие на doc_0:")
for doc_tag, similarity in similar_docs:
    doc_id = int(doc_tag.split('_')[1])
    print(f"  {doc_tag}: {similarity:.4f}")
    print(f"    Текст: {documents[doc_id]}")

Вектор документа doc_0: [-0.01057    -0.01198188 -0.01982618  0.01710627  0.00710373]...

Документы, похожие на doc_0:
  doc_1: 0.2735
    Текст: deep learning uses neural networks
  doc_2: 0.1275
    Текст: python programming for data science


In [26]:
# Сравниваем схожесть документов
def compare_documents(doc1_id, doc2_id):
    similarity = doc_model.dv.similarity(f"doc_{doc1_id}", f"doc_{doc2_id}")
    print(f"Схожесть doc_{doc1_id} и doc_{doc2_id}: {similarity:.4f}")
    print(f"  doc_{doc1_id}: {documents[doc1_id]}")
    print(f"  doc_{doc2_id}: {documents[doc2_id]}")

compare_documents(0, 1)  # machine learning vs deep learning
compare_documents(0, 3)  # machine learning vs AI

Схожесть doc_0 и doc_1: 0.2735
  doc_0: machine learning is interesting
  doc_1: deep learning uses neural networks
Схожесть doc_0 и doc_3: -0.0822
  doc_0: machine learning is interesting
  doc_3: artificial intelligence is amazing


8. Сравните схожесть doc_2 и doc_4

In [27]:
compare_documents(2, 4)

Схожесть doc_2 и doc_4: -0.0362
  doc_2: python programming for data science
  doc_4: computer vision processes images


9. Найдите самый похожий документ на doc_1

In [28]:
most_similar_doc = doc_model.dv.most_similar('doc_1', topn=1)

print("Самый похожий документ на doc_1:")

for doc_tag, similarity in most_similar_doc:

    doc_id = int(doc_tag.split('_')[1])

    print(f"  {doc_tag}: {similarity:.4f}")
    print(f"  Текст: {documents[doc_id]}")

Самый похожий документ на doc_1:
  doc_0: 0.2735
  Текст: machine learning is interesting


10. Выберите любую из трёх моделей. Обучите модели с разной размерностью (10, 50, 100). Продемонстрируйте качество их работы на примере поиска похожих слов (выберите любые 3 примера, соответствующих тематике корпуса из пункта 4)

In [29]:
vector_sizes = [10, 50, 100]

test_words = ['овощи', 'тесто', 'мясо']

for size in vector_sizes:

    print(f"\n{'=' * 50}")
    print(f"Размерность векторов: {size}")

    temp_model = Word2Vec(
        sentences=cooking_sentences,
        vector_size=size,
        window=3,
        min_count=1,
        workers=2,
        sg=1
    )

    for word in test_words:

        print(f"\nПохожие слова для '{word}':")

        similar_words = temp_model.wv.most_similar(word, topn=3)

        for similar_word, score in similar_words:
            print(f"  {similar_word}: {score:.4f}")


Размерность векторов: 10

Похожие слова для 'овощи':
  жарить: 0.7185
  фольга: 0.7146
  горшок: 0.6696

Похожие слова для 'тесто':
  специи: 0.6146
  тушить: 0.6120
  барбекю: 0.5774

Похожие слова для 'мясо':
  жарить: 0.6996
  курица: 0.6485
  мука: 0.5814

Размерность векторов: 50

Похожие слова для 'овощи':
  мариновать: 0.2716
  хлеб: 0.2691
  гриль: 0.2546

Похожие слова для 'тесто':
  жарить: 0.2416
  масло: 0.2372
  помидоры: 0.2005

Похожие слова для 'мясо':
  завтрак: 0.3760
  взбивать: 0.2301
  жарить: 0.2296

Размерность векторов: 100

Похожие слова для 'овощи':
  взбивать: 0.2190
  питание: 0.2163
  бекон: 0.1956

Похожие слова для 'тесто':
  десерт: 0.2529
  месить: 0.2009
  горшок: 0.1957

Похожие слова для 'мясо':
  тост: 0.1890
  кипятить: 0.1885
  помидоры: 0.1839
